# Proyecto final — Pipeline completo

**Tiempo estimado:** 1 h 30 min.

**Objetivo.** Aplicar todo lo aprendido en el curso a uno de los dos datasets de referencia y entregar un **notebook + breve reporte** comparando modelos estadísticos, ML y DL.

## Reglas del juego

1. Elige uno de los dos datasets:
   - **A. Caudal del Genil** (Pinos-Genil, ROEA 5020) — predicción a 7 días con foco en la crecida.
   - **B. Cota piezométrica `PZ0267014`** (Renedo de Esgueva, Valladolid) — predicción a 1 mes con foco en bajadas por extracción.

2. Implementa al menos **3 modelos** de **familias distintas** (un estadístico, un ML clásico, opcional un DL).

3. Evalúa con **NSE + KGE + error en pico** (o error en bajada para piezometría).

4. Cierra con **conclusiones honestas**: qué funcionó, qué no, qué intentarías con más tiempo.

## Criterios de evaluación

| Bloque | Peso |
|---|---|
| Carga + EDA correcta (sin fuga, splits temporales) | 25% |
| Implementación de los modelos (mínimo 3 familias) | 30% |
| Evaluación con métricas adecuadas | 20% |
| Análisis de errores (al menos por régimen) | 15% |
| Conclusiones y reflexión | 10% |

El instructor está disponible para dudas. **Reuse libremente** los notebooks 1-4 del curso.

## 0 · Setup (no tocar)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../sesion1'))
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import utils_datos as ud

plt.rcParams.update({'figure.figsize': (10, 3.4), 'axes.grid': True, 'grid.alpha': 0.3})

# Métricas hidrológicas (reutilizadas en todo el curso)
def nse(o, s):
    o, s = np.asarray(o), np.asarray(s)
    return 1 - np.sum((o - s) ** 2) / np.sum((o - o.mean()) ** 2)

def kge(o, s):
    o, s = np.asarray(o), np.asarray(s)
    r = np.corrcoef(o, s)[0, 1]
    return 1 - np.sqrt((r-1)**2 + (s.std()/o.std() - 1)**2 + (s.mean()/o.mean() - 1)**2)

## 1 · Elección del dataset

Edita la siguiente celda para elegir **A** o **B**.

In [ ]:
OPCION = 'A'   # 'A' = caudal Genil  |  'B' = piezometría PZ0267014

if OPCION == 'A':
    serie = ud.cargar_caudal_genil().loc['1995':'2020']
    lluvia = ud.cargar_lluvia_genil_diaria(fecha_inicio='1995-01-01', fecha_fin='2020-12-31')
    HORIZONTE_DIAS = 7
    descripcion = 'Caudal diario en Pinos-Genil (ROEA 5020), horizonte 7 días.'
elif OPCION == 'B':
    serie = ud.cargar_piezometria('PZ0267014')
    lluvia = ud.cargar_lluvia_duero_diaria(fecha_inicio='1985-01-01', fecha_fin='2024-12-31')
    HORIZONTE_DIAS = 30
    descripcion = 'Cota piezométrica PZ0267014 (Valladolid), horizonte 1 mes.'
else:
    raise ValueError('OPCION debe ser "A" o "B"')

print(descripcion)
print(ud.resumen(serie))

## 2 · EDA — exploración

**TODO** (mínimo):

- Plot serie completa.
- Plot zoom al evento foco (crecida o bajada por extracción).
- ACF / PACF.
- STL si la serie es regular; estadísticos descriptivos por mes/estación si es irregular.
- Reportar **cobertura temporal** y **gaps**.

In [ ]:
# TODO: EDA. Inspírate en notebook 1 (sesion1/01_carga_y_exploracion.ipynb).
fig, ax = plt.subplots()
ax.plot(serie.index, serie.values, color='#1f6f8b', lw=0.5)
ax.set_title('Serie completa')
plt.tight_layout()

## 3 · Splits temporales

**Train, val, test** cronológicos. Decide y justifica las fechas según la cobertura.

In [ ]:
# TODO: definir splits
if OPCION == 'A':
    SPLIT_TRAIN_END = pd.Timestamp('2015-12-31')
    SPLIT_VAL_END   = pd.Timestamp('2017-12-31')
    SPLIT_TEST_END  = pd.Timestamp('2020-12-31')
else:  # B
    SPLIT_TRAIN_END = pd.Timestamp('2010-12-31')
    SPLIT_VAL_END   = pd.Timestamp('2015-12-31')
    SPLIT_TEST_END  = pd.Timestamp('2024-11-12')

print(f'Train  → {SPLIT_TRAIN_END.date()}')
print(f'Val    → {SPLIT_VAL_END.date()}')
print(f'Test   → {SPLIT_TEST_END.date()}')

## 4 · Modelo 1 — Baseline estadístico

**TODO:** Elige uno y justifica:

- Opción A: SARIMAX (notebook 2.1)
- Opción A: Holt-Winters (ETS) (notebook 2.1)
- Opción A: Prophet (notebook 2.2)
- **Opción B: Pastas** (notebook 2.3) — recomendado para piezometría.

In [ ]:
# TODO: Modelo 1
pred_modelo1 = None   # tu predicción sobre el test
obs_modelo1  = None   # tu observado sobre el test
# Asegúrate de que ambos están alineados temporalmente

## 5 · Modelo 2 — ML clásico

**TODO:** RF, XGBoost o LightGBM con `skforecast`. Usa las features del notebook 3.1 (lags + rolling + calendario + lluvia).

In [ ]:
# TODO: Modelo 2
pred_modelo2 = None
obs_modelo2  = None

## 6 · Modelo 3 (opcional) — Deep Learning

**TODO opcional:** LSTM con Keras (notebook 4.1).

In [ ]:
# TODO: Modelo 3 (opcional)
pred_modelo3 = None
obs_modelo3  = None

## 7 · Tabla comparativa

**TODO:** Rellenar.

In [ ]:
# TODO: tabla comparativa con NSE, KGE, RMSE, MAE, error en pico (o en bajada)
filas = []
for nombre, obs, pred in [
    ('Modelo 1', obs_modelo1, pred_modelo1),
    ('Modelo 2', obs_modelo2, pred_modelo2),
    ('Modelo 3', obs_modelo3, pred_modelo3),
]:
    if pred is None: continue
    o, p = np.asarray(obs), np.asarray(pred)
    filas.append({
        'modelo': nombre,
        'NSE': round(nse(o, p), 3),
        'KGE': round(kge(o, p), 3),
        'RMSE': round(float(np.sqrt(((o - p) ** 2).mean())), 3),
        'error_pico': round(float(p.max() - o.max()), 3),
    })
pd.DataFrame(filas).set_index('modelo')

## 8 · Análisis de errores

**TODO:** Mínimo: errores por régimen (estiaje vs crecida o subida vs bajada). Inspírate en notebook 4.3.

In [ ]:
# TODO: análisis de errores por régimen

## 9 · Conclusiones

Responde a estas tres preguntas en 1-2 párrafos cada una:

### 9.1 ¿Qué modelo recomendarías para este problema y por qué?

**TODO:** texto.

### 9.2 ¿En qué casos fallaría tu modelo recomendado? ¿Cómo lo detectarías?

**TODO:** texto.

### 9.3 ¿Qué intentarías con más tiempo / más datos?

**TODO:** texto.

---

*Entrega:* sube este notebook completado al canal del curso. Si no terminas en clase, dispones de 1 semana para enviarlo.